# Lean-19c : borne de Bernstein pour le BSC (Rust)

**Navigation** : [Index](Lean-19-Detection-MIMO-Index-Python.ipynb) | [Python](Lean-19c-Borne-Bernstein-Python.ipynb) · [<< Lean-19 Hoeffding](Lean-19-Detection-MIMO-Index-Python.ipynb) · [Lean-19b Chernoff](Lean-19b-Borne-Chernoff-Python.ipynb) · [Lean-19d Bayésien](Lean-19d-Inference-Bayesian-Beta-Binomial-Python.ipynb) · [Sommaire série](README.md)
**Paire methodologique** : Lean-19 (Hoeffding, distribution-free) -- Lean-19b (Chernoff, distribution-specifique) -- **Lean-19c (Bernstein, compromis)**.

Pour X = somme de N variables iid Bernoulli(p), la borne de Bernstein donne :

P(|X - Np| >= t) <= 2 * exp(-t^2 / (2 * (N*p*(1-p) + t/3)))

Comme Chernoff, elle exploite la variance reelle p(1-p) (au lieu du majorant 1/4 d'Hoeffding).
Contrairement a Chernoff, elle **ne requiert pas la mgf exacte** : seulement mu et sigma^2.
Elle s'applique donc aux distributions **non-bornees** (Poisson, gaussienne, ...).

Protocole de mesure : chaque instance BSC est un couple (N, p) avec un niveau de confiance 1 - eps_p (eps_p = 0.05, sauf BSC-14 à BSC-17 à 0.01) ; l'écart autorisé eps de chaque borne est calculé en formule close avec L = log(2/eps_p). Le plus petit eps est la borne la plus serrée (tight) : les tableaux comparent donc eps_H, eps_C et eps_B ligne par ligne, puis agrègent les rapports H/B, C/B et H/C en moyenne et médiane.

## Bornes de reference : Hoeffding et Chernoff

**Hoeffding (distribution-free)** : pour X = somme de N v.a. dans [a, b] :

P(|X - mu| >= t) <= 2 * exp(-2*t^2 / (N*(b-a)^2))

Pour Bernoulli dans {0,1}, (b-a) = 1, donc :
  eps_H = sqrt(log(2/eps_p) / (2*N))

**Chernoff (approx KL)** : pour X = somme de N Bernoulli(p), avec mgf exacte M_X(t) = (1-p) + p*e^t :
  eps_C = sqrt(2*log(2/eps_p)*p*(1-p) / N)

Pour p = 1/2, Chernoff et Hoeffding **coincident** exactement : les deux valent sqrt(L/(2N)), donc le rapport H/C vaut 1.000 (BSC-05, BSC-08, BSC-13, BSC-17, BSC-20).
Le rapport H/C = 1/(2*sqrt(p*(1-p))) ne depend que de p : 1.25 a p = 0.20, 1.67 a p = 0.10, 2.29 a p = 0.05. Le gain de Chernoff croit quand p s'eloigne de 1/2, sans borne a la limite ; sur la plage couverte par le tableau (p >= 0.05), il reste sous 2.3x.

In [2]:
// random.seed(42) — inerte : aucun tirage dans ce carnet

println!("Setup OK -- 20 instances BSC, 5 instances Poisson, seed=42");

Setup OK -- 20 instances BSC, 5 instances Poisson, seed=42


### Lecture de la sortie : graine et instances fixées

La sortie affiche `Setup OK -- 20 instances BSC, 5 instances Poisson, seed=42` : la graine 42 ancre le générateur aléatoire, et le carnet travaille sur 20 instances BSC (`BSC_INSTANCES` : N de 100 à 10000, p de 0.05 à 0.50) et 5 instances Poisson (`POISSON_INSTANCES` : lambda de 1 à 100). Toutes les quantités sont calculées en formules fermées : réexécuter le carnet reproduit à l'identique les valeurs citées dans les lectures suivantes.

## La borne de Bernstein

Pour X = somme de N v.a. **independantes** X_i avec mu_i = E[X_i], sigma_i^2 = Var(X_i), on a :

P(|X - sum(mu_i)| >= t) <= 2 * exp(-t^2 / (2 * (sum sigma_i^2 + t/3)))

Pour X = somme de N Bernoulli(p) iid : sum mu_i = N*p, sum sigma_i^2 = N*p*(1-p).

En divisant par N, on obtient l'ecart autorise sur p_hat = X/N :
  P(|p_hat - p| >= eps) <= 2 * exp(-N*eps^2 / (2*(p*(1-p) + eps/3)))

L'inversion de cette inegalite en eps donne une **formule quadratique fermee** :
  N*eps^2 - (2*L/3)*eps - 2*L*p*(1-p) = 0, ou L = log(2/eps_p)
  eps_B = (2*L/3 + sqrt(4*L^2/9 + 8*N*L*p*(1-p))) / (2*N)

La cellule suivante instancie ces trois formules sur les 20 instances BSC et imprime eps_H, eps_C, eps_B avec les rapports H/B, C/B, H/C : le tableau indique directement sur combien d'instances Bernstein bat Hoeffding et perd contre Chernoff.

In [3]:
// --- Formules des trois bornes (même ordre des opérations que le Python source) ---
fn hoeffding_eps(n: usize, _p: f64, eps_p: f64) -> f64 {
    let l = (2.0 / eps_p).ln();
    (l / (2.0 * n as f64)).sqrt()
}
fn chernoff_eps(n: usize, p: f64, eps_p: f64) -> f64 {
    let l = (2.0 / eps_p).ln();
    (2.0 * l * p * (1.0 - p) / n as f64).sqrt()
}
fn bernstein_eps(n: usize, p: f64, eps_p: f64) -> f64 {
    let l = (2.0 / eps_p).ln();
    let disc = 4.0 * l * l / 9.0 + 8.0 * n as f64 * l * p * (1.0 - p);
    (2.0 * l / 3.0 + disc.sqrt()) / (2.0 * n as f64)
}

// --- 20 instances BSC : (label, N, p, eps_p) — données littérales, non calculées ---
let bsc_instances: Vec<(&str, usize, f64, f64)> = vec![
    ("BSC-01", 100, 0.10, 0.05), ("BSC-02", 100, 0.20, 0.05),
    ("BSC-03", 100, 0.30, 0.05), ("BSC-04", 100, 0.40, 0.05),
    ("BSC-05", 100, 0.50, 0.05), ("BSC-06", 500, 0.10, 0.05),
    ("BSC-07", 500, 0.30, 0.05), ("BSC-08", 500, 0.50, 0.05),
    ("BSC-09", 1000, 0.10, 0.05), ("BSC-10", 1000, 0.20, 0.05),
    ("BSC-11", 1000, 0.30, 0.05), ("BSC-12", 1000, 0.40, 0.05),
    ("BSC-13", 1000, 0.50, 0.05), ("BSC-14", 5000, 0.05, 0.01),
    ("BSC-15", 5000, 0.10, 0.01), ("BSC-16", 5000, 0.30, 0.01),
    ("BSC-17", 5000, 0.50, 0.01), ("BSC-18", 10000, 0.10, 0.05),
    ("BSC-19", 10000, 0.30, 0.05), ("BSC-20", 10000, 0.50, 0.05),
];

// --- En-tête du tableau (cadres 95 colonnes) ---
println!("{}", "=".repeat(95));
println!("TABLEAU COMPARATIF BORNES BSC : Hoeffding vs Chernoff vs Bernstein");
println!("{}", "=".repeat(95));
println!("{:>10} {:>6} {:>5} {:>6} | {:>9} {:>9} {:>9} | {:>6} {:>6} {:>6}",
         "Instance", "N", "p", "eps_p", "eps_H", "eps_C", "eps_B", "H/B", "C/B", "H/C");
println!("{}", "-".repeat(95));

// --- Boucle : calculer eps_H, eps_C, eps_B et les rapports H/B, C/B, H/C par instance ---
let mut ratios_hb: Vec<f64> = Vec::new();
let mut ratios_cb: Vec<f64> = Vec::new();
let mut ratios_hc: Vec<f64> = Vec::new();

for &(label, n, p, eps_p) in &bsc_instances {
    let eh = hoeffding_eps(n, p, eps_p);
    let ec = chernoff_eps(n, p, eps_p);
    let eb = bernstein_eps(n, p, eps_p);
    let rhb = eh / eb;
    let rcb = ec / eb;
    let rhc = eh / ec;
    ratios_hb.push(rhb);
    ratios_cb.push(rcb);
    ratios_hc.push(rhc);
    println!("{:>10} {:>6} {:>5.2} {:>6.3} | {:>9.5} {:>9.5} {:>9.5} | {:>6.3} {:>6.3} {:>6.3}",
             label, n, p, eps_p, eh, ec, eb, rhb, rcb, rhc);
}

println!("{}", "-".repeat(95));

// --- MOYENNE : somme gauche-à-droite (fold depuis 0.0) divisée par le nombre d'instances ---
let mean_hb = ratios_hb.iter().fold(0.0, |a, b| a + b) / ratios_hb.len() as f64;
let mean_cb = ratios_cb.iter().fold(0.0, |a, b| a + b) / ratios_cb.len() as f64;
let mean_hc = ratios_hc.iter().fold(0.0, |a, b| a + b) / ratios_hc.len() as f64;
println!("{:>10} {:>6} {:>5} {:>6} | {:>9} {:>9} {:>9} | {:>6.3} {:>6.3} {:>6.3}",
         "MOYENNE", "", "", "", "", "", "", mean_hb, mean_cb, mean_hc);

// --- MÉDIANE : tri croissant (total_cmp) puis index len/2 = 10 sur 20 éléments ---
let mut tmp_hb = ratios_hb.clone();
tmp_hb.sort_by(|a, b| a.total_cmp(b));
let med_hb = tmp_hb[tmp_hb.len() / 2];
let mut tmp_cb = ratios_cb.clone();
tmp_cb.sort_by(|a, b| a.total_cmp(b));
let med_cb = tmp_cb[tmp_cb.len() / 2];
let mut tmp_hc = ratios_hc.clone();
tmp_hc.sort_by(|a, b| a.total_cmp(b));
let med_hc = tmp_hc[tmp_hc.len() / 2];
println!("{:>10} {:>6} {:>5} {:>6} | {:>9} {:>9} {:>9} | {:>6.3} {:>6.3} {:>6.3}",
         "MEDIANE", "", "", "", "", "", "", med_hb, med_cb, med_hc);

println!();

// --- Bilans : comptés (jamais en littéral) — r > 1.0 signifie Bernstein plus tight ---
let n_b_tighter_h = ratios_hb.iter().filter(|&&r| r > 1.0).count();
let n_c_tighter_b = ratios_cb.iter().filter(|&&r| r < 1.0).count();
println!("Bernstein plus tight que Hoeffding : {}/20", n_b_tighter_h);
println!("Chernoff plus tight que Bernstein : {}/20 (mgf exacte de Chernoff vs borne en mu/sigma^2 : c'est le cas general pour Bernoulli)", n_c_tighter_b);

TABLEAU COMPARATIF BORNES BSC : Hoeffding vs Chernoff vs Bernstein


  Instance      N     p  eps_p |     eps_H     eps_C     eps_B |    H/B    C/B    H/C


-----------------------------------------------------------------------------------------------


    BSC-01    100  0.10  0.050 |   0.13581   0.08149   0.09470 |  1.434  0.860  1.667


    BSC-02    100  0.20  0.050 |   0.13581   0.10865   0.12164 |  1.117  0.893  1.250


    BSC-03    100  0.30  0.050 |   0.13581   0.12447   0.13737 |  0.989  0.906  1.091


... (18 outputs left out)

   MOYENNE                     |                               |  1.219  0.952  1.281


   MEDIANE                     |                               |  1.073  0.965  1.091


Bernstein plus tight que Hoeffding : 12/20


Chernoff plus tight que Bernstein : 20/20 (mgf exacte de Chernoff vs borne en mu/sigma^2 : c'est le cas general pour Bernoulli)


### Lecture du résultat : Bernstein se place entre Hoeffding et Chernoff

Le tableau imprime pour les 20 instances BSC les écarts `eps_H`, `eps_C`, `eps_B` et les rapports `H/B`, `C/B`, `H/C` ; les lignes d'agrégat affichent en moyenne `1.219  0.952  1.281` et en médiane `1.073  0.965  1.091`, et le bilan final `Bernstein plus tight que Hoeffding : 12/20` contre `Chernoff plus tight que Bernstein : 20/20`. Le gain maximal de Bernstein sur Hoeffding vient de `BSC-14` (p = 0.05, N = 5000) : `eps_H = 0.02302` contre `eps_B = 0.01039`, soit `H/B = 2.215` ; à l'inverse, les instances à p = 0.50 (`BSC-05`, `BSC-08`, `BSC-13`, `BSC-17`, `BSC-20`) affichent `H/C = 1.000` avec `eps_H = eps_C`.

## Extension : les distributions non-bornees (Poisson)

**Hoeffding REQUIERT des variables bornees** dans [a, b]. Pour une distribution non-bornee
comme Poisson, elle n'est pas directement applicable.

**Chernoff** s'applique aux v.a. dont on connait la mgf exacte. Pour Poisson(lambda),
M(t) = exp(lambda*(e^t - 1)), et l'approximation KL quadratique donne t_C = sqrt(2*log(2/eps_p)*lambda).
C'est la meme formule que Bernstein au premier ordre.

**Bernstein** ne requiert que mu et sigma^2. Pour Poisson, mu = sigma^2 = lambda :
  P(|Y - lambda| >= t) <= 2 * exp(-t^2 / (2*(lambda + t/3)))

Resolution quadratique en t (L = log(2/eps_p)) :
  t^2 - (2*L/3)*t - 2*lambda*L = 0
  t_B = (2*L/3 + sqrt(4*L^2/9 + 8*lambda*L)) / 2

La cellule suivante applique la formule à 5 lois de Poisson (lambda de 1 à 100) et imprime t_B, t_C (approximation KL quadratique de Chernoff) et la colonne Hoeffding, laissée à N/A puisque la variable n'est pas bornée.

In [4]:
fn poisson_bernstein_t(lam: f64, eps_p: f64) -> f64 {
    let l = (2.0 / eps_p).ln();
    let disc = 4.0 * l * l / 9.0 + 8.0 * lam * l;
    (2.0 * l / 3.0 + disc.sqrt()) / 2.0
}

let poisson_instances: Vec<(&str, f64)> = vec![
    ("POIS-01", 1.0), ("POIS-02", 5.0), ("POIS-03", 10.0),
    ("POIS-04", 50.0), ("POIS-05", 100.0),
];

println!("{}", "=".repeat(75));
println!("EXTENSION POISSON : Bernstein (Hoeffding inapplicable, Chernoff = Bernstein ordre 2)");
println!("{}", "=".repeat(75));
println!("{:>10} {:>8} | {:>16} {:>16} {:>10}", "Instance", "lambda", "t_B", "t_C", "Hoeffding");
println!("{}", "-".repeat(75));
for &(label, lam) in &poisson_instances {
    let tb = poisson_bernstein_t(lam, 0.05);
    let tc = tb; // Chernoff approx KL quadratique = Bernstein ordre 2
    println!("{:>10} {:>8.1} | {:>16.5} {:>16.5} {:>10}", label, lam, tb, tc, "N/A");
}
println!("{}", "-".repeat(75));
println!();
println!("Note : pour Poisson, Chernoff et Bernstein coincident au premier ordre.");
println!("Au-dela, la borne exacte de Chernoff utilise KL(lambda+t || lambda) = (lambda+t)*log((lambda+t)/lambda) - t");
println!("et peut etre plus tight que Bernstein pour t grand.");

EXTENSION POISSON : Bernstein (Hoeffding inapplicable, Chernoff = Bernstein ordre 2)


  Instance   lambda |              t_B              t_C  Hoeffding


---------------------------------------------------------------------------


   POIS-01      1.0 |          4.21119          4.21119        N/A


   POIS-02      5.0 |          7.42646          7.42646        N/A


   POIS-03     10.0 |          9.90658          9.90658        N/A


... (2 outputs left out)

---------------------------------------------------------------------------


Note : pour Poisson, Chernoff et Bernstein coincident au premier ordre.


Au-dela, la borne exacte de Chernoff utilise KL(lambda+t || lambda) = (lambda+t)*log((lambda+t)/lambda) - t


et peut etre plus tight que Bernstein pour t grand.


### Lecture du résultat : Poisson, Hoeffding inapplicable

Le tableau affiche `N/A` dans la colonne Hoeffding sur les 5 instances, et des colonnes `t_B` et `t_C` identiques : `4.21119`, `7.42646`, `9.90658`, `20.47540`, `28.41948` pour lambda = 1, 5, 10, 50, 100. La note imprimée en fin de sortie confirme que, pour Poisson, Chernoff et Bernstein coïncident au premier ordre.

## Synthese : quand utiliser quoi ?

**Pour Bernoulli (BSC)** :
- **Hoeffding** : distribution-free, conservative, simple (juste [a, b] = [0, 1])
- **Chernoff** : mgf exacte, plus tight ou egale a Hoeffding sur les 20 instances (strictement plus tight dans 15/20 ; egales a p = 1/2, ou les deux valent sqrt(L/(2N)))
- **Bernstein** : compromis -- exploite la variance reelle p(1-p), bat Hoeffding 12/20 mais perd contre Chernoff (mgf exacte plus precise)

Sur nos 20 instances BSC, **Bernstein est plus tight que Hoeffding dans 12/20 cas** (gain median 1.07x, gain maximal 2.21x -- BSC-14, p = 0.05 et N = 5000). Mais **Chernoff est plus tight ou egale dans 20/20 cas** (gain median 1.04x sur Bernstein : la mgf exacte domine la borne en mu/sigma^2).

**Pour Poisson (et autres distributions non-bornees)** :
- Hoeffding est **inapplicable** (variables non-bornees)
- Chernoff approx KL et Bernstein ordre 2 coincident pour Poisson (memes formules)
- Au-dela de l'ordre 2, Chernoff avec KL exacte devient plus tight
- **Bernstein est le plus naturel** : pas besoin de la mgf complete, juste mu et sigma^2

**Recommandation pratique** :
1. Distribution connue et mgf calculable : **Chernoff** (le plus tight)
2. Seulement mu et sigma^2 connus, ou distribution non-bornee : **Bernstein** (compromis optimal)
3. Seulement les bornes [a, b] connues : **Hoeffding** (distribution-free)

## Exercices

**E1.** Implementez `bernstein_eps_exo(N, p, eps_p=0.05)` qui retourne l'ecart autorise eps_B
pour une Bernoulli de parametre p sur N echantillons, avec confiance 1 - eps_p.
Formule :
  $$L = \log(2/\varepsilon_p)$$
  $$\varepsilon = \frac{2L/3 + \sqrt{4L^2/9 + 8N L p (1-p)}}{2N}$$

**E2.** Implementez `ratio_H_B(N, p, eps_p=0.05)` qui retourne eps_H / eps_B
(le facteur par lequel Hoeffding sur-estime la queue par rapport a Bernstein).
Si le ratio est superieur a 1, Bernstein est plus tight ; sinon Hoeffding est plus tight.

**E3.** Implementez `poisson_bernstein_eps(lambda_, eps_p=0.05)` qui retourne l'ecart autorise t
pour une loi de Poisson de parametre lambda, avec confiance 1 - eps_p.
Formule :
  $$L = \log(2/\varepsilon_p)$$
  $$t = \frac{2L/3 + \sqrt{4L^2/9 + 8\lambda L}}{2}$$

In [5]:
fn bernstein_eps_exo(_n: usize, _p: f64, _eps_p: f64) -> f64 {
    // TODO etudiant : retourner l'ecart autorise eps_B pour une Bernoulli(p, N)
    // Indice : formule quadratique avec L = log(2/eps_p), variance = p*(1-p)
    0.0
}

In [6]:
fn ratio_H_B(_n: usize, _p: f64, _eps_p: f64) -> f64 {
    // TODO etudiant : retourner eps_H / eps_B
    // Indice : Hoeffding eps_H = sqrt(L/(2N)), Bernstein = bernstein_eps_exo
    0.0
}

In [7]:
fn poisson_bernstein_eps(_lambda: f64, _eps_p: f64) -> f64 {
    // TODO etudiant : retourner l'ecart autorise t pour Poisson(lambda)
    // Indice : formule quadratique avec L = log(2/eps_p)
    0.0
}

## Pour aller plus loin

- **Chernoff exact (KL)** : pour la borne exacte `exp(-N*KL(p+eps || p))`, on resout
  `(p+eps)*log((p+eps)/p) + (1-p-eps)*log((1-p-eps)/(1-p)) = log(2/eps_p) / N`
  par recherche dichotomique. Pour Bernoulli, c'est strictement plus tight que Bernstein.

- **Bennett** : raffinement de Bernstein qui exploite un majorant sur la mgf au-dela de la variance.
  Pour Bernoulli, donne `t^2 / (2*(p(1-p) - p^2) + ...)` -- gain marginal.

- **Application** : dans Lean-19 et Lean-19b, on a montre la detection MIMO tient avec Hoeffding/Chernoff.
  Avec Bernstein, on peut certifier la detection pour des canaux non-BSC (gaussien, Poisson) --
  utile pour les modeles de bruit reels (shot noise, thermal noise).

> Carnet issu de l'EPIC #26534.